## Animation of The Frequency of $\varpi$ As $\alpha$ Varies

In [6]:
import matplotlib.pyplot as plt
import numpy as np
import rebound
from scipy import signal
from PIL import Image 
import glob
import os

In [2]:
#setting up the range Saturn's semi-major axis so that alpha varries using Kepler's Law
a_sat = np.linspace(5.2*(4.3**(2/3)),5.2*(1.73**(3/2)),100)

In [9]:
#looping through each sim
for i in a_sat:
    sim = rebound.Simulation()
    sim.integrator = "whfast"
    sim.dt = 6e-2
    sim.add(m=1,hash='Sun')
    sim.add(m=9.945786e-4,a=5.2,e=0.03,inc = 0,Omega = 0,pomega=0.5*np.pi,hash="Jupiter")
    sim.add(m=2.85837e-4,a=i,e=0.04,inc = 0, Omega = 0,pomega=0,hash="Saturn")
    sim.move_to_com()

    times = np.linspace(0,1e6,10000)
    times = times*2*np.pi
    long_peri_sat = np.zeros(len(times))
    e_sat = np.zeros(len(times))
    for j in range(len(times)):
        sim.integrate(times[j])
        long_peri_sat[j] = sim.particles[2].pomega
        e_sat[j] = sim.particles[2].e

    
    
    h = e_sat*np.sin(long_peri_sat)
    k = e_sat*np.cos(long_peri_sat)
    z = k + h*1j

    fs = 1/((times[1] - times[0])/(2*np.pi))

    f, pxx = signal.periodogram(h,fs=fs,window='hann',scaling='density')

    plt.stem(f*2*np.pi,pxx,markerfmt = "none")
    plt.xscale('log')
    plt.ylabel('Power (With Power Spectral Density Scaling)')
    plt.xlabel(r"Frequency of the longitude of perihelion $\frac{radians}{year}$")
    plt.xlim(1e-7,1e-2)
    plt.title(rf'Power Spectral Density vs Frequency of Longitude of Perihelion when $\alpha$ = {i}')
    plt.savefig(f'gif_graphs/graph_num_{np.where(a_sat == i)[0]}',bbox_inches='tight')
    plt.close()

In [10]:
#making the gif

fp_in = "gif_graphs/graph_num_*.png"
fp_out = "freq_of_peri.gif"
img, *imgs = [Image.open(f) for f in sorted(glob.glob(fp_in))]

img.save(fp=fp_out, format='GIF', append_images=imgs,save_all=True, duration=200,loop=0)      

